# Focus Guard v3 — built from scratch

A new model, not a port. v2 ported the Keras mini-Xception and plateaued after three
runs at **0.662 / 0.764** (7-class / 4-state macro-F1) on `data/test`, with training loss
at 0.57 while validation stopped moving around epoch 30 — classic overfitting against a
noisy-label ceiling.

| Run | best val F1_4 | test F1_4 |
|---|---|---|
| v2 run A | 0.7506 | 0.7648 |
| v2 run B | 0.7473 | 0.7641 |
| **v3 target** | — | **> 0.79** |

Same seed produced 0.7506 and 0.7473, so the **noise floor is about ±0.004**. Any change
smaller than that is unproven without multiple seeds. This notebook answers that by
training several seeds and ensembling them.

## What changes, and why

| Change | Reason |
|---|---|
| VGG-style deep CNN | Best published FER2013 single net (73.28%) is a tuned VGGNet. Depth with 3x3 filters beats separable convs, which exist to save mobile FLOPs you aren't short of |
| SGD + Nesterov + cosine | AdamW fit the training set too fast. SGD generalises better for from-scratch CNNs |
| mixup | Attacks the exact overfitting the v2 curves showed |
| Random erasing | Forces the model off single cues (one eye, the mouth corner) — occlusion robustness the webcam will meet |
| EMA weights | Averaged weights beat final weights; roughly free |
| N-seed ensemble | Turns the ±0.004 run variance into a real gain |
| 4-state selection + cost-aware calibration | Unchanged from v2 — still the metric the app consumes |

Leave `SMOKE_TEST = True` for the first pass.

In [ ]:
import os, sys, math, glob, json, time, copy
import numpy as np

PROJECT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(PROJECT); sys.path.insert(0, PROJECT)

IMG        = 48
BATCH      = 128          # smaller batch + SGD generalises better than 256 + Adam
EPOCHS     = 120
SEEDS      = [1337, 7, 2024]      # one model per seed, then ensembled
ARCH       = 'vgg13'      # 'vgg13' (9.4M) or 'vgg16' (14.7M)
LR         = 0.05         # SGD; scaled for batch 128
WEIGHT_DECAY = 5e-4
MIXUP_ALPHA  = 0.2
ERASE_P      = 0.25
EMA_DECAY    = 0.999
W_FALSE_STRESS = 0.25     # knee of the measured frontier: halves false alarms for ~1 point of F1
SMOKE_TEST   = True

CLAHE_DIR = 'data_clahe'
CKPT_FMT  = 'src/models/focus_guard_v3_seed{}.pt'
ONNX      = 'src/models/focus_guard_v3.onnx'

CLASSES  = ['angry','disgust','fear','happy','neutral','sad','surprise']
STATES   = ['FOCUS','HAPPY','STRESS','DISTRACTION']
STATE_ID = {'neutral':0,'happy':1,'angry':2,'disgust':2,'fear':2,'sad':2,'surprise':3}
CLASS_TO_STATE = np.array([STATE_ID[c] for c in CLASSES])

V2 = {'acc7': 0.6620, 'acc4': 0.7779, 'f1_4': 0.7641,
      'false_stress': 0.261, 'missed_stress': 0.222}

if SMOKE_TEST:
    EPOCHS, SEEDS = 2, [1337]
print(f'arch {ARCH} | epochs {EPOCHS} | seeds {SEEDS} | smoke {SMOKE_TEST}')

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__)
assert torch.cuda.is_available(), 'No GPU visible to torch'
DEVICE = torch.device('cuda')
print('gpu  ', torch.cuda.get_device_name(0), '| capability', '.'.join(map(str, torch.cuda.get_device_capability())))
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
print('amp  ', AMP_DTYPE)

## 1. Data

Reuses `data_clahe/` from the v2 notebook — the CLAHE-matched copy that makes training
images look like what the webcam sends. Build it there first if it's missing.

Augmentation is heavier than v2: random erasing joins the affine jitter, and mixup is
applied on-device during training.

In [ ]:
from torch.utils.data import DataLoader, Subset
from torchvision import transforms as T
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split

assert os.path.isdir(CLAHE_DIR), 'run the CLAHE cell in focus_guard_torch.ipynb first'

train_tf = T.Compose([
    T.Grayscale(num_output_channels=1),
    T.RandomHorizontalFlip(),
    T.RandomAffine(degrees=12, translate=(0.1, 0.1), scale=(0.88, 1.12)),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.RandomErasing(p=ERASE_P, scale=(0.02, 0.15)),      # occlusion robustness
])
eval_tf = T.Compose([T.Grayscale(num_output_channels=1), T.ToTensor()])

root = CLAHE_DIR + '/train'
ds_tr_view, ds_ev_view = ImageFolder(root, train_tf), ImageFolder(root, eval_tf)
assert ds_tr_view.classes == CLASSES, ds_tr_view.classes

targets = np.array(ds_tr_view.targets)
idx = np.arange(len(targets))
if SMOKE_TEST:
    idx, _ = train_test_split(idx, train_size=2000, stratify=targets, random_state=0)
tr_idx, va_idx = train_test_split(idx, test_size=0.1, stratify=targets[idx], random_state=0)

kw = dict(num_workers=8, pin_memory=True, persistent_workers=True)
train_dl = DataLoader(Subset(ds_tr_view, tr_idx), batch_size=BATCH, shuffle=True, drop_last=True, **kw)
val_dl   = DataLoader(Subset(ds_ev_view, va_idx), batch_size=512, shuffle=False, **kw)
test_dl  = DataLoader(ImageFolder(CLAHE_DIR + '/test', eval_tf), batch_size=512, shuffle=False, **kw)

counts  = np.bincount(targets[tr_idx], minlength=7).astype(np.float64)
class_w = torch.tensor((counts.sum() / (7 * counts)) ** 0.5, dtype=torch.float32, device=DEVICE)
print('train', len(tr_idx), '| val', len(va_idx), '| test', len(test_dl.dataset))
print('weights', {CLASSES[i]: round(float(v), 2) for i, v in enumerate(class_w.cpu())})

## 2. Architecture — VGG-style for 48x48

Five 3x3 stages, each halving resolution: 48 → 24 → 12 → 6 → 3 → 1. No fully-connected
pyramid at the end (that's where classic VGG spent 100M parameters); global pooling plus
one linear layer instead.

Depth is the point. Stacked 3x3 convolutions build a large receptive field out of small
filters, which is what a 48px face needs — the difference between a furrowed brow and a
raised one is a few pixels of texture, not a large-scale shape.

In [ ]:
CFG = {
    'vgg13': [64,64,'M', 128,128,'M', 256,256,'M', 512,512,'M', 512,512,'M'],
    'vgg16': [64,64,'M', 128,128,'M', 256,256,256,'M', 512,512,512,'M', 512,512,512,'M'],
}

class VGGFace(nn.Module):
    def __init__(self, arch='vgg13', num_classes=7, dropout=0.5):
        super().__init__()
        layers, cin = [], 1
        for v in CFG[arch]:
            if v == 'M':
                layers.append(nn.MaxPool2d(2, 2))
            else:
                layers += [nn.Conv2d(cin, v, 3, padding=1, bias=False),
                           nn.BatchNorm2d(v), nn.ReLU(inplace=True)]
                cin = v
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(dropout), nn.Linear(cin, num_classes))
        for m in self.modules():                       # He init: the right prior for ReLU nets
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.head(self.features(x))

_m = VGGFace(ARCH).to(DEVICE)
print(ARCH, round(sum(p.numel() for p in _m.parameters())/1e6, 2), 'M params')
print('output', tuple(_m(torch.zeros(2,1,IMG,IMG, device=DEVICE)).shape))
del _m

## 3. Training

**mixup** blends two images and their labels: the network never sees a clean example
twice, so it cannot memorise the training set the way v2 did.

**EMA** keeps a slowly-moving average of the weights. SGD bounces around the minimum; the
average sits closer to its centre and usually scores better than any single step.

Checkpoints are selected on 4-state macro-F1 — the metric the app consumes.

In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from torch.optim.swa_utils import AveragedModel, get_ema_avg_fn

@torch.no_grad()
def probs_of(net, dl, tta=True):
    net.eval(); P, Y = [], []
    for xb, yb in dl:
        xb = xb.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', dtype=AMP_DTYPE):
            p = net(xb).float().softmax(1)
            if tta:
                p = (p + net(torch.flip(xb, dims=[3])).float().softmax(1)) / 2
        P.append(p.cpu()); Y.append(yb)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def f1_4(probs, y):
    return float(f1_score(CLASS_TO_STATE[y], CLASS_TO_STATE[probs.argmax(1)], average='macro'))

def train_one(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    net = VGGFace(ARCH).to(DEVICE).to(memory_format=torch.channels_last)
    ema = AveragedModel(net, avg_fn=get_ema_avg_fn(EMA_DECAY), use_buffers=True)
    crit = nn.CrossEntropyLoss(weight=class_w, label_smoothing=0.05)
    opt = torch.optim.SGD(net.parameters(), lr=LR, momentum=0.9,
                          weight_decay=WEIGHT_DECAY, nesterov=True)
    warm = max(1, round(EPOCHS * 0.05))
    sched = torch.optim.lr_scheduler.SequentialLR(opt, [
        torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.05, total_iters=warm),
        torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS-warm), eta_min=1e-5)],
        milestones=[warm])

    path, best = CKPT_FMT.format(seed), -1.0
    for ep in range(EPOCHS):
        net.train(); t0, tot, n = time.time(), 0.0, 0
        for xb, yb in train_dl:
            xb = xb.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
            yb = yb.to(DEVICE, non_blocking=True)
            lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)) if MIXUP_ALPHA > 0 else 1.0
            perm = torch.randperm(xb.size(0), device=DEVICE)
            xb = lam * xb + (1 - lam) * xb[perm]                    # mixup
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=AMP_DTYPE):
                out = net(xb)
                loss = lam * crit(out, yb) + (1 - lam) * crit(out, yb[perm])
            loss.backward(); opt.step(); ema.update_parameters(net)
            tot += loss.item() * xb.size(0); n += xb.size(0)
        sched.step()

        raw_f1 = f1_4(*probs_of(net, val_dl, tta=False))
        ema_f1 = f1_4(*probs_of(ema.module, val_dl, tta=False))
        use_ema = ema_f1 >= raw_f1
        cur = max(raw_f1, ema_f1)
        flag = ''
        if cur > best:
            best = cur
            torch.save({'state_dict': (ema.module if use_ema else net).state_dict(),
                        'arch': ARCH, 'classes': CLASSES, 'f1_4': best, 'ema': use_ema}, path)
            flag = f"  <- best ({'ema' if use_ema else 'raw'}), saved"
        print(f"  seed {seed} ep {ep+1:3d}/{EPOCHS} loss {tot/n:.4f} "
              f"raw {raw_f1:.4f} ema {ema_f1:.4f} {time.time()-t0:5.1f}s{flag}")
    print(f'seed {seed}: best val 4-state macro-F1 {best:.4f} -> {path}')
    return path, best

results = [train_one(s) for s in SEEDS]

## 4. Ensemble

Averaging softmax across seeds cancels the per-run variance we measured (±0.004) instead
of being at its mercy. Each member saw the same data in a different order with different
mixup draws and different initial weights, so their mistakes are partly independent.

In [ ]:
def load_model(path):
    ck = torch.load(path, map_location=DEVICE)
    net = VGGFace(ck['arch']).to(DEVICE)
    net.load_state_dict(ck['state_dict']); net.eval()
    return net, ck

members = []
for path, _ in results:
    net, ck = load_model(path)
    members.append(net)
    print(f"{path}  val F1_4 {ck['f1_4']:.4f}  ({'ema' if ck['ema'] else 'raw'} weights)")

class Ensemble(nn.Module):
    def __init__(self, nets):
        super().__init__(); self.nets = nn.ModuleList(nets)
    def forward(self, x):                       # returns probabilities, not logits
        return torch.stack([n(x).softmax(1) for n in self.nets]).mean(0)

ens = Ensemble(members).to(DEVICE).eval()

@torch.no_grad()
def ens_probs(dl, tta=True):
    P, Y = [], []
    for xb, yb in dl:
        xb = xb.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', dtype=AMP_DTYPE):
            p = ens(xb).float()
            if tta:
                p = (p + ens(torch.flip(xb, dims=[3])).float()) / 2
        P.append(p.cpu()); Y.append(yb)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

for i, net in enumerate(members):
    print(f'member {i} val F1_4 (TTA) {f1_4(*probs_of(net, val_dl)):.4f}')
print('ENSEMBLE  val F1_4 (TTA)', round(f1_4(*ens_probs(val_dl)), 4))

## 5. Cost-aware calibration

Identical to v2: state probabilities get a per-state bias fitted on validation. Macro-F1
alone treats interrupting a focused user and missing stress as equally bad; the app does
not. `W_FALSE_STRESS` is that asymmetry as a dial, and the sweep shows the exchange rate.

In [ ]:
def to_state_probs(p):
    s = np.zeros((len(p), 4))
    for c in range(7):
        s[:, CLASS_TO_STATE[c]] += p[:, c]
    return s

def decide(sp, bias):
    return (np.log(sp + 1e-9) + bias).argmax(1)

def app_metrics(sp, ys, bias):
    pred = decide(sp, bias)
    calm, stressed = (ys == 0), (ys == 2)
    return {'f1': float(f1_score(ys, pred, average='macro')),
            'acc': float((pred == ys).mean()),
            'false_stress':  float((calm & (pred == 2)).sum() / max(1, calm.sum())),
            'missed_stress': float((stressed & (pred != 2)).sum() / max(1, stressed.sum()))}

def fit_bias(sp, ys, w, sweeps=3):
    grid, bias = np.arange(-1.0, 1.01, 0.05), np.zeros(4)
    for _ in range(sweeps):
        for k in range(4):
            sc = []
            for g in grid:
                m = app_metrics(sp, ys, np.where(np.arange(4) == k, g, bias))
                sc.append(m['f1'] - w * m['false_stress'])
            bias[k] = grid[int(np.argmax(sc))]
    return bias

vp, vy_ = ens_probs(val_dl)
vs, vy = to_state_probs(vp), CLASS_TO_STATE[vy_]

print(f"{'w':>4}  {'bias [FOCUS HAPPY STRESS DISTRACT]':36s} {'F1':>7} {'acc':>7} "
      f"{'false-stress':>13} {'missed-stress':>14}")
frontier = {}
for w in sorted({0.0, 0.25, 0.5, 1.0, 2.0, float(W_FALSE_STRESS)}):
    b = fit_bias(vs, vy, w); m = app_metrics(vs, vy, b); frontier[w] = b
    print(f"{w:4.2f}  {str(np.round(b,2)):36s} {m['f1']:7.4f} {m['acc']:7.4f} "
          f"{m['false_stress']:13.3f} {m['missed_stress']:14.3f}")
bias = frontier[float(W_FALSE_STRESS)]
print('\nusing W_FALSE_STRESS =', W_FALSE_STRESS, '->', dict(zip(STATES, bias.round(2))))

## 6. Test — v3 against v2

In [ ]:
tp, ty_ = ens_probs(test_dl)
ts, ty = to_state_probs(tp), CLASS_TO_STATE[ty_]
raw, cal = app_metrics(ts, ty, np.zeros(4)), app_metrics(ts, ty, bias)
acc7 = float((tp.argmax(1) == ty_).mean())

print(classification_report(ty_, tp.argmax(1), target_names=CLASSES, digits=3, zero_division=0))
print('4-state confusion after calibration', STATES)
print(confusion_matrix(ty, decide(ts, bias)))
print()
print(f"{'':26s} {'7-class':>8} {'F1_4':>8} {'acc_4':>8} {'false-str':>10} {'missed-str':>11}")
print(f"{'v2 (mini-Xception)':26s} {V2['acc7']:8.4f} {V2['f1_4']:8.4f} {V2['acc4']:8.4f} "
      f"{V2['false_stress']:10.3f} {V2['missed_stress']:11.3f}")
print(f"{'v3 ensemble':26s} {acc7:8.4f} {raw['f1']:8.4f} {raw['acc']:8.4f} "
      f"{raw['false_stress']:10.3f} {raw['missed_stress']:11.3f}")
print(f"{'v3 ensemble + calibrated':26s} {acc7:8.4f} {cal['f1']:8.4f} {cal['acc']:8.4f} "
      f"{cal['false_stress']:10.3f} {cal['missed_stress']:11.3f}")
print(f"\ndelta vs v2 on 4-state macro-F1: {cal['f1'] - V2['f1_4']:+.4f}  (noise floor is ~0.004)")

os.makedirs('runs', exist_ok=True)
json.dump({'v2': V2, 'v3_raw': raw, 'v3_calibrated': cal, 'acc7': acc7, 'arch': ARCH,
           'seeds': SEEDS, 'state_bias': bias.tolist(), 'w_false_stress': W_FALSE_STRESS},
          open('runs/scores_v3.json', 'w'), indent=2)

## 7. Export the ensemble as one ONNX file

The whole ensemble exports as a single graph that returns averaged probabilities — the
Mac sees one model and one call, with no idea it's three networks.

In [ ]:
meta = {'classes': CLASSES, 'states': STATES, 'input': [1, 1, IMG, IMG],
        'state_bias': bias.tolist(), 'arch': ARCH, 'members': len(members),
        'preprocessing': 'GaussianBlur(3,3) -> CLAHE(2.0, 8x8) -> resize 48x48 -> /255'}
json.dump(meta, open(ONNX + '.json', 'w'), indent=2)

try:
    cpu_ens = Ensemble([copy.deepcopy(m).cpu().eval() for m in members]).eval()
    torch.onnx.export(cpu_ens, torch.zeros(1, 1, IMG, IMG), ONNX,
                      input_names=['input'], output_names=['probs'],
                      dynamic_axes={'input': {0: 'batch'}, 'probs': {0: 'batch'}},
                      external_data=False)   # one self-contained file, not a .onnx + .onnx.data pair
    print('wrote', ONNX, round(os.path.getsize(ONNX)/1e6, 2), 'MB')

    # Prove the exported graph matches the trained ensemble before shipping it.
    try:
        import onnxruntime as ort
        xb, _ = next(iter(test_dl))
        x = xb[:8].numpy().astype(np.float32)
        ref = ens(torch.from_numpy(x).to(DEVICE)).detach().cpu().numpy()
        got = ort.InferenceSession(ONNX, providers=['CPUExecutionProvider']).run(None, {'input': x})[0]
        print('onnx vs torch, max abs diff:', float(np.abs(ref - got).max()), '(want < 1e-5)')
    except ImportError:
        print('(install onnxruntime to verify the export numerically)')

    print('scp', ONNX, ONNX + '.json', ' mac:<project>/src/models/')
except ModuleNotFoundError as e:
    print('ONNX export needs onnxscript:', e, '\n  uv pip install onnxscript')